In [ ]:
import os
from pathlib import Path
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2

BASE_DIR = Path("/content/drive/MyDrive")

CLEAN_CSV = BASE_DIR / "pig-selected_all-image" / "behavior_clean_merged.csv"

IMG_ROOT  = BASE_DIR / "pig-selected_all-image" / "images_clean"

ROI_COCO_JSON = Path("/content/_annotations.coco.json")

# File output sau khi build feature
OUT_FEATS_CSV = BASE_DIR / "pig-selected_all-image" / "behavior_with_feats_rectROI.csv"

print("CLEAN_CSV:", CLEAN_CSV)
print("IMG_ROOT :", IMG_ROOT)
print("ROI_COCO:", ROI_COCO_JSON)

assert CLEAN_CSV.exists()
assert ROI_COCO_JSON.exists()
assert IMG_ROOT.exists()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def polygon_from_segmentation(seg):
    pts = []
    arr = list(seg)
    assert len(arr) % 2 == 0
    for i in range(0, len(arr), 2):
        pts.append((float(arr[i]), float(arr[i+1])))
    return pts

def load_roi_from_coco_auto(coco_path):
    with open(coco_path, "r") as f:
        coco = json.load(f)

    if not coco.get("images"):
        raise ValueError()

    img0 = coco["images"][0]
    roi_w = float(img0["width"])
    roi_h = float(img0["height"])
    print(f"[ROI] base size from COCO: {roi_w} x {roi_h}")

    cat_id_to_name = {c["id"]: c["name"] for c in coco.get("categories", [])}

    rois_poly = defaultdict(list)  # name -> [poly...]

    for ann in coco.get("annotations", []):
        cname = cat_id_to_name.get(ann["category_id"], None)
        if cname not in ("feeder", "drinker", "toy"):
            continue
        segs = ann.get("segmentation", [])
        if not segs:
            continue
        if isinstance(segs[0], (int, float)):
            poly = polygon_from_segmentation(segs)
            rois_poly[cname].append(poly)
        else:
            for seg in segs:
                poly = polygon_from_segmentation(seg)
                rois_poly[cname].append(poly)

    print("[ROI] polygons loaded:")
    for k, v in rois_poly.items():
        print(f"  {k}: {len(v)} polygon(s)")

    return rois_poly, roi_w, roi_h

ROIS_POLY, ROI_W, ROI_H = load_roi_from_coco_auto(ROI_COCO_JSON)


In [ ]:
sample_img_path = None
for fn in os.listdir(IMG_ROOT):
    if fn.lower().endswith((".jpg", ".jpeg", ".png")):
        sample_img_path = os.path.join(IMG_ROOT, fn)
        break

assert sample_img_path is not None
img = cv2.imread(sample_img_path)
H_img, W_img = img.shape[:2]
print("Sample image size (W,H):", W_img, H_img)

scale_x = W_img / ROI_W
scale_y = H_img / ROI_H
print("scale_x, scale_y:", scale_x, scale_y)

def scaled_roi_bboxes(rois_poly, sx, sy, margin=5.0):
    """
    """
    rois_box = defaultdict(list)
    for name, polys in rois_poly.items():
        for poly in polys:
            xs = [p[0] * sx for p in poly]
            ys = [p[1] * sy for p in poly]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            x1 -= margin
            y1 -= margin
            x2 += margin
            y2 += margin

            rois_box[name].append((x1, y1, x2, y2))
    return rois_box

ROI_BOXES = scaled_roi_bboxes(ROIS_POLY, scale_x, scale_y, margin=8.0)

print("[ROI] bboxes:")
for k, v in ROI_BOXES.items():
    print(f"  {k}: {v}")


In [ ]:
def rect_intersect(a, b):
    """
    a, b: (x1,y1,x2,y2)
    return: area_intersection
    """
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    return (ix2 - ix1) * (iy2 - iy1)

df = pd.read_csv(CLEAN_CSV)
print("Cols:", df.columns.tolist())
print("Rows:", len(df))

needed_cols = ["img_name", "behavior", "x1", "y1", "x2", "y2", "group_id", "pig_id", "order"]
missing = [c for c in needed_cols if c not in df.columns]
if missing:
    raise ValueError()

# center + size
df["cx"] = (df["x1"] + df["x2"]) / 2.0
df["cy"] = (df["y1"] + df["y2"]) / 2.0
df["bw"] = df["x2"] - df["x1"]
df["bh"] = df["y2"] - df["y1"]

df["cx_n"] = df["cx"] / W_img
df["cy_n"] = df["cy"] / H_img
df["bw_n"] = df["bw"] / W_img
df["bh_n"] = df["bh"] / H_img

in_feeder, in_drinker, in_toy = [], [], []

for _, row in df.iterrows():
    pig_box = (float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"]))

    f_flag = 0
    for rb in ROI_BOXES.get("feeder", []):
        if rect_intersect(pig_box, rb) > 0:
            f_flag = 1
            break

    d_flag = 0
    for rb in ROI_BOXES.get("drinker", []):
        if rect_intersect(pig_box, rb) > 0:
            d_flag = 1
            break

    t_flag = 0
    for rb in ROI_BOXES.get("toy", []):
        if rect_intersect(pig_box, rb) > 0:
            t_flag = 1
            break

    in_feeder.append(f_flag)
    in_drinker.append(d_flag)
    in_toy.append(t_flag)

df["in_feeder"]  = in_feeder
df["in_drinker"] = in_drinker
df["in_toy"]     = in_toy

print("[ROI flags sum]")
print(df[["in_feeder","in_drinker","in_toy"]].sum())
print(df[["in_feeder","in_drinker","in_toy"]].head())


In [ ]:
diag = math.sqrt(W_img**2 + H_img**2)

speed_feat = np.zeros(len(df), dtype="float32")

for (gid, pid), sub in df.groupby(["group_id", "pig_id"]):
    sub = sub.sort_values("order")
    idxs = sub.index.to_list()
    prev_cx, prev_cy = None, None
    for i, idx in enumerate(idxs):
        cx = df.at[idx, "cx"]
        cy = df.at[idx, "cy"]
        if prev_cx is None:
            s = 0.0
        else:
            dist = math.sqrt((cx - prev_cx)**2 + (cy - prev_cy)**2)
            s = dist / (diag + 1e-9)
        speed_feat[idx] = s
        prev_cx, prev_cy = cx, cy

df["speed_feat"] = speed_feat
print("[speed_feat] min/max:", df["speed_feat"].min(), df["speed_feat"].max())


In [ ]:
min_dist_other = np.zeros(len(df), dtype="float32")
num_close_other = np.zeros(len(df), dtype="float32")

CLOSE_THRESH = 0.12

for (gid, order), sub in df.groupby(["group_id", "order"]):
    idxs = sub.index.to_list()
    coords = sub[["cx","cy"]].values
    n = len(idxs)
    if n <= 1:
        for idx in idxs:
            min_dist_other[idx] = 0.0
            num_close_other[idx] = 0.0
        continue

    for i in range(n):
        idx_i = idxs[i]
        cx_i, cy_i = coords[i]
        dists = []
        for j in range(n):
            if i == j:
                continue
            cx_j, cy_j = coords[j]
            dist = math.sqrt((cx_i-cx_j)**2 + (cy_i-cy_j)**2) / (diag + 1e-9)
            dists.append(dist)

        if dists:
            min_d = min(dists)
            cnt_close = sum(d < CLOSE_THRESH for d in dists)
        else:
            min_d = 0.0
            cnt_close = 0

        min_dist_other[idx_i] = min_d
        num_close_other[idx_i] = float(cnt_close)

df["min_dist_other"] = min_dist_other
df["num_close_other"] = num_close_other

print(df[["min_dist_other","num_close_other"]].describe())


In [ ]:
FINE_BEHAVIORS = [
    "drink",
    "eat",
    "fight",
    "social-nose",
    "explore",
    "lying",
    "stand",
    "move",
    "sitting",
    "playwithtoy",
]

MERGE_MAP = {
    "lying"       : "resting",
    "sitting"     : "resting",

    "eat"         : "feeding",
    "drink"       : "feeding",

    "move"        : "locomotion",
    "stand"       : "locomotion",
    "explore"     : "locomotion",
    "playwithtoy" : "locomotion",

    "social-nose" : "social",
    "fight"       : "social",
}

df["behavior"] = df["behavior"].astype(str)
df["behavior_coarse"] = df["behavior"].map(MERGE_MAP)

before = len(df)
df = df[df["behavior_coarse"].notna()].reset_index(drop=True)
after = len(df)
print(f"[COARSE] kept {after}/{before} rows")

print("Fine behavior dist:")
print(df["behavior"].value_counts())
print("\nCoarse behavior dist:")
print(df["behavior_coarse"].value_counts())

df.to_csv(OUT_FEATS_CSV, index=False)
print("[SAVE] features CSV:", OUT_FEATS_CSV)
print(df[["in_feeder","in_drinker","in_toy"]].sum())


In [ ]:
df_roi = pd.read_csv(OUT_FEATS_CSV)
print(df_roi[["behavior","in_feeder","in_drinker","in_toy"]]
      .groupby("behavior")[["in_feeder","in_drinker","in_toy"]]
      .mean())


In [ ]:
import cv2

def polygon_from_segmentation(seg):
    """
    return: list[(x,y), ...]
    """
    pts = []
    arr = list(seg)
    assert len(arr) % 2 == 0
    for i in range(0, len(arr), 2):
        pts.append((float(arr[i]), float(arr[i+1])))
    return pts

def point_in_poly(x, y, poly):
    """
    poly: list[(x,y), ...]
    """
    inside = False
    n = len(poly)
    if n < 3:
        return False
    px, py = x, y
    x1, y1 = poly[0]
    for i in range(n+1):
        x2, y2 = poly[i % n]
        if min(y1, y2) < py <= max(y1, y2) and px <= max(x1, x2):
            if y2 != y1:
                xinters = (py - y1) * (x2 - x1) / (y2 - y1 + 1e-9) + x1
            else:
                xinters = x1
            if xinters >= px:
                inside = not inside
        x1, y1 = x2, y2
    return inside

def load_roi_polygons_from_coco(coco_path):
    """
      {
        "feeder": [poly1, poly2,...],
        "drinker": [...],
        "toy": [...]
      }
    """
    with open(coco_path, "r") as f:
        coco = json.load(f)

    # mapping category_id -> name
    cat_id_to_name = {c["id"]: c["name"] for c in coco.get("categories", [])}

    rois = defaultdict(list)

    for ann in coco.get("annotations", []):
        cat_id = ann["category_id"]
        cat_name = cat_id_to_name.get(cat_id, None)
        if cat_name not in ("feeder", "drinker", "toy"):
            continue
        segs = ann.get("segmentation", [])
        if not segs:
            continue
        if isinstance(segs[0], (int, float)):
            poly = polygon_from_segmentation(segs)
            rois[cat_name].append(poly)
        else:
            for seg in segs:
                poly = polygon_from_segmentation(seg)
                rois[cat_name].append(poly)

    print("[ROI] Loaded polygons:")
    for k, v in rois.items():
        print(f"  {k}: {len(v)} polygon(s)")
    return rois

ROIS = load_roi_polygons_from_coco(ROI_COCO_JSON)


In [ ]:
import random
import matplotlib.pyplot as plt

def scaled_rois(rois, sx, sy):
    """
    """
    out = {}
    for name, polys in rois.items():
        new_polys = []
        for poly in polys:
            new_poly = [(x*sx, y*sy) for (x,y) in poly]
            new_polys.append(new_poly)
        out[name] = new_polys
    return out
ROIS = load_roi_polygons_from_coco(ROI_COCO_JSON)
SCALED_ROIS = scaled_rois(ROIS, scale_x, scale_y)
cand = df[df["behavior"].isin(["eat","drink","playwithtoy"])].copy()
cand = cand.sample(n=1, random_state=0)
row = cand.iloc[0]

img_path = os.path.join(IMG_ROOT, row["img_name"])
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8,5))
plt.imshow(img)

x1,y1,x2,y2 = row["x1"],row["y1"],row["x2"],row["y2"]
plt.gca().add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                  fill=False, edgecolor="lime", linewidth=2))
cx, cy = row["cx"], row["cy"]
plt.scatter([cx],[cy], c="yellow", s=40)

for name, polys in SCALED_ROIS.items():
    for poly in polys:
        xs = [p[0] for p in poly] + [poly[0][0]]
        ys = [p[1] for p in poly] + [poly[0][1]]
        color = {"feeder":"red","drinker":"blue","toy":"orange"}.get(name,"white")
        plt.plot(xs, ys, color=color, linewidth=2)

plt.title(f"{row['img_name']} | beh={row['behavior']} (f={row['in_feeder']},d={row['in_drinker']},t={row['in_toy']})")
plt.axis("off")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive")
FEATS_CSV = BASE_DIR / "pig-selected_all-image" / "behavior_with_feats_rectROI.csv"
IMG_ROOT  = BASE_DIR / "pig-selected_all-image" / "images_clean"

print("FEATS_CSV:", FEATS_CSV)
print("IMG_ROOT :", IMG_ROOT)

assert FEATS_CSV.exists()
assert IMG_ROOT.exists()

df_feat = pd.read_csv(FEATS_CSV)
print("Rows:", len(df_feat))
print("Columns:", df_feat.columns.tolist())

# ====== BEHAVIOR COARSE ======
COARSE_BEHAVIORS = ["resting", "feeding", "locomotion", "social"]
coarse2idx = {b:i for i,b in enumerate(COARSE_BEHAVIORS)}
idx2coarse = {i:b for b,i in coarse2idx.items()}
NUM_CLASSES = len(COARSE_BEHAVIORS)

assert "behavior_coarse" in df_feat.columns

df_feat = df_feat[df_feat["behavior_coarse"].isin(COARSE_BEHAVIORS)].reset_index(drop=True)
print("After coarse filter rows:", len(df_feat))
print(df_feat["behavior_coarse"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

df_feat["group_id"] = df_feat["group_id"].astype(str)
df_feat["pig_id"]   = df_feat["pig_id"].astype(str)
df_feat["behavior_coarse"] = df_feat["behavior_coarse"].astype(str)

seq_rows = []
for (gid, pid), sub in df_feat.groupby(["group_id", "pig_id"]):
    behaviors = sub["behavior_coarse"].value_counts()
    maj = behaviors.idxmax()
    seq_rows.append({
        "group_id": gid,
        "pig_id": pid,
        "behavior_coarse": maj,
        "n_frames": len(sub),
    })

seq_df = pd.DataFrame(seq_rows)
print("Total sequences:", len(seq_df))
print("Sequence behavior dist:")
print(seq_df["behavior_coarse"].value_counts())

# Encode label
seq_df["label_idx"] = seq_df["behavior_coarse"].map(coarse2idx)

# Split train/val theo sequence
labels_all = seq_df["label_idx"].values
indices = np.arange(len(seq_df))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=labels_all
)

seq_df_train = seq_df.iloc[train_idx].reset_index(drop=True)
seq_df_val   = seq_df.iloc[val_idx].reset_index(drop=True)

print("Train seq:", len(seq_df_train))
print("Val   seq:", len(seq_df_val))
print("Train coarse dist:")
print(seq_df_train["behavior_coarse"].value_counts())
print("Val coarse dist:")
print(seq_df_val["behavior_coarse"].value_counts())

seq_df_train["split"] = "train"
seq_df_val["split"]   = "val"
seq_split = pd.concat([seq_df_train, seq_df_val], ignore_index=True)[["group_id","pig_id","split"]]

df_feat = df_feat.merge(seq_split, on=["group_id","pig_id"], how="inner")
print("Frame-level rows after merge & split:", len(df_feat))
print(df_feat["split"].value_counts())


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np

class BehaviorSequenceDataset(Dataset):
    def __init__(self, df, img_root, behavior2idx, split="train", transform=None):
        """
            img_name, group_id, pig_id, order,
            x1,y1,x2,y2,
            cx_n,cy_n,bw_n,bh_n,
            speed_feat, min_dist_other, num_close_other,
            in_feeder, in_drinker, in_toy,
            behavior_coarse, split
        behavior2idx: mapping coarse behavior -> index
        """
        self.img_root = str(img_root)
        self.behavior2idx = behavior2idx
        self.transform = transform

        df = df.copy()
        df = df[df["split"] == split].reset_index(drop=True)

        self.groups = []
        for (gid, pid), sub in df.groupby(["group_id","pig_id"]):
            if "order" in sub.columns:
                sub = sub.sort_values("order")
            else:
                sub = sub.sort_values("img_name")

            beh = sub["behavior_coarse"].iloc[0]
            if beh not in behavior2idx:
                continue

            self.groups.append({
                "group_id": gid,
                "pig_id": pid,
                "behavior": beh,
                "rows": sub.reset_index(drop=True),
            })

        print(f"[SEQ DATASET] {split}: {len(self.groups)} sequences")

        self.extra_cols = [
            "cx_n", "cy_n", "bw_n", "bh_n",
            "speed_feat",
            "min_dist_other", "num_close_other",
            "in_feeder", "in_drinker", "in_toy",
        ]
        self.BURST_LEN = 6

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        rows = g["rows"]
        T = len(rows)

        imgs = []
        feats = []

        for i in range(T):
            row = rows.iloc[i]
            img_path = os.path.join(self.img_root, row["img_name"])
            img = Image.open(img_path).convert("RGB")

            # crop bbox
            x1, y1, x2, y2 = row["x1"], row["y1"], row["x2"], row["y2"]
            img = img.crop((x1, y1, x2, y2))

            if self.transform is not None:
                img_t = self.transform(img)
            else:
                arr = np.array(img).astype("float32") / 255.0
                arr = np.transpose(arr, (2, 0, 1))
                img_t = torch.from_numpy(arr)

            feat = torch.tensor(row[self.extra_cols].values.astype("float32"))

            imgs.append(img_t)
            feats.append(feat)

        while len(imgs) < self.BURST_LEN:
            imgs.append(imgs[-1].clone())
            feats.append(feats[-1].clone())
        imgs = imgs[:self.BURST_LEN]
        feats = feats[:self.BURST_LEN]

        imgs = torch.stack(imgs, dim=0)    # [T,C,H,W]
        feats = torch.stack(feats, dim=0)  # [T,K]

        label = torch.tensor(self.behavior2idx[g["behavior"]], dtype=torch.long)
        return imgs, feats, label


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        # Channel attention
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        # Spatial attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=spatial_kernel,
                                      padding=spatial_kernel // 2, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        # Channel
        avg_pool = F.adaptive_avg_pool2d(x, 1).view(B, C)
        max_pool = F.adaptive_max_pool2d(x, 1).view(B, C)
        ca = self.mlp(avg_pool) + self.mlp(max_pool)
        ca = torch.sigmoid(ca).view(B, C, 1, 1)
        x = x * ca

        # Spatial
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.cat([avg_map, max_map], dim=1)
        sa = torch.sigmoid(self.conv_spatial(sa))
        x = x * sa
        return x

class BehaviorTransformerNet(nn.Module):
    def __init__(
        self,
        num_behaviors,
        extra_dim,
        d_model=256,
        nhead=4,
        num_layers=2,
        backbone_name="resnet18",
        freeze_backbone=False,
        dropout=0.2,
    ):
        super().__init__()

        # Backbone
        if backbone_name == "resnet18":
            backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        elif backbone_name == "resnet34":
            backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        else:
            raise ValueError("Unsupported backbone")

        self.cnn = nn.Sequential(*list(backbone.children())[:-2])  # conv5_x output
        cnn_out_channels = backbone.fc.in_features  # 512 cho resnet18/34

        if freeze_backbone:
            for p in self.cnn.parameters():
                p.requires_grad = False

        # CBAM
        self.cbam = CBAMBlock(cnn_out_channels, reduction=16, spatial_kernel=7)

        base_dim = d_model - extra_dim
        if base_dim <= 0:
            raise ValueError()
        self.cnn_proj = nn.Linear(cnn_out_channels, base_dim)

        # Temporal Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.attn_fc = nn.Linear(d_model, 1)

        # Head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_behaviors),
        )

    def forward(self, seq_imgs, seq_feats):
        """
        seq_imgs: [B,T,C,H,W]
        seq_feats: [B,T,K]
        """
        B, T, C, H, W = seq_imgs.shape
        x = seq_imgs.view(B*T, C, H, W)

        feat_map = self.cnn(x)          # [B*T,C',H',W']
        feat_map = self.cbam(feat_map)
        feat_vec = F.adaptive_avg_pool2d(feat_map, 1).view(B*T, -1)  # [B*T,C']

        feat_vec = self.cnn_proj(feat_vec)  # [B*T, base_dim]
        feat_vec = feat_vec.view(B, T, -1)  # [B,T,base_dim]

        if seq_feats is not None:
            seq_feats = seq_feats.to(feat_vec.dtype)
            h = torch.cat([feat_vec, seq_feats], dim=-1)  # [B,T,d_model]
        else:
            h = feat_vec

        h_enc = self.transformer(h)  # [B,T,d_model]

        attn_score = self.attn_fc(h_enc).squeeze(-1)   # [B,T]
        attn_weight = torch.softmax(attn_score, dim=1) # [B,T]
        attn_weight = attn_weight.unsqueeze(-1)        # [B,T,1]

        z = torch.sum(h_enc * attn_weight, dim=1)      # [B,d_model]
        logits = self.head(z)                          # [B,num_behaviors]
        return logits


In [ ]:
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# ====== TRANSFORMS ======
train_tf = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.2),
    T.ToTensor(),
])

val_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# ====== DATASET & LOADER ======
train_ds = BehaviorSequenceDataset(df_feat, IMG_ROOT, coarse2idx, split="train", transform=train_tf)
val_ds   = BehaviorSequenceDataset(df_feat, IMG_ROOT, coarse2idx, split="val",   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

extra_dim = len(train_ds.extra_cols)
print("extra_dim:", extra_dim, "num_classes:", NUM_CLASSES)

# ====== MODEL / LOSS / OPTIM ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# class weights (theo sequence)
train_counts = seq_df_train["behavior_coarse"].value_counts().reindex(COARSE_BEHAVIORS, fill_value=0)
freq = train_counts / train_counts.sum()
weights_cls = 1.0 / np.sqrt(freq.values + 1e-8)
class_weights = torch.tensor(weights_cls, dtype=torch.float32).to(device)
print("Train counts:", train_counts.to_dict())
print("Class weights:", class_weights.cpu().numpy())

model = BehaviorTransformerNet(
    num_behaviors=NUM_CLASSES,
    extra_dim=extra_dim,
    d_model=256,
    nhead=4,
    num_layers=2,
    backbone_name="resnet34",
    freeze_backbone=False,
    dropout=0.2,
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

USE_AMP = True
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

NUM_EPOCHS = 50
PATIENCE   = 7
max_grad_norm = 5.0

best_macroF1 = 0.0
no_improve   = 0

# file checkpoint
CHECKPOINT_PATH = Path("/content/drive/MyDrive/pig_behavior_coarse_best.pt")

for epoch in range(1, NUM_EPOCHS+1):
    print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} =====")

    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for seq_imgs, seq_feats, labels in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        seq_imgs = seq_imgs.to(device)
        seq_feats = seq_feats.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(seq_imgs, seq_feats)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        # gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= max(train_total, 1)
    train_acc = train_correct / max(train_total, 1)

    # ---- VAL ----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_labels = []
    all_preds  = []

    with torch.inference_mode():
        for seq_imgs, seq_feats, labels in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
            seq_imgs = seq_imgs.to(device)
            seq_feats = seq_feats.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(seq_imgs, seq_feats)
                loss = criterion(logits, labels)

            val_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    val_loss /= max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)
    macroF1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch} | train_loss={train_loss:.4f} acc={train_acc:.3f} "
          f"| val_loss={val_loss:.4f} acc={val_acc:.3f} macroF1={macroF1:.3f}")

    if macroF1 > best_macroF1 + 1e-4:
        best_macroF1 = macroF1
        no_improve   = 0
        print(f"  -> New best macroF1={best_macroF1:.3f}, saving checkpoint to {CHECKPOINT_PATH}")
        torch.save(model.state_dict(), CHECKPOINT_PATH)
    else:
        no_improve += 1
        print(f"  -> No improvement for {no_improve} epoch(s) (best macroF1={best_macroF1:.3f})")

    if no_improve >= PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs (patience={PATIENCE}).")
        break

print("\nTraining finished.")
print("Best val macro-F1:", best_macroF1)

if CHECKPOINT_PATH.exists():
    print(f"Loading best model from {CHECKPOINT_PATH}")
    state = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(state)
    model.to(device)


In [ ]:
model.eval()
all_labels = []
all_preds  = []

with torch.inference_mode():
    for seq_imgs, seq_feats, labels in tqdm(val_loader, desc="Final eval"):
        seq_imgs = seq_imgs.to(device)
        seq_feats = seq_feats.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(seq_imgs, seq_feats)
        preds = logits.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

print("Final validation classification report (coarse):")
print(classification_report(
    all_labels,
    all_preds,
    target_names=COARSE_BEHAVIORS,
    digits=3,
))
